In [1]:
# Wariant 14: porownanie metryk i wybor modelu pod zastosowanie
# Cwiczenie laboratoryjne 12: badania przypadkow klinicznych.
# Cele:
# - porownanie modeli LR, RF i Gradient Boosting,
# - ocena dla progu 0.5 i progu dopasowanego do Precision = 0.80,
# - wybor modelu do screeningu i diagnostyki poglebionej,
# - analiza 5 bledow krytycznych FN.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, roc_auc_score, average_precision_score
    )
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

np.random.seed(42)
pd.set_option("display.max_columns", 200)

In [3]:
# 1) Wczytanie danych
DATA_PATH = "cases_clinical_for_lab12.csv"
target_col = "high_risk_cvd"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

print("Braki danych (%):")
display((df.isna().mean() * 100).sort_values(ascending=False).round(2))

print("Rozkład klasy docelowej:")
display(df[target_col].value_counts(dropna=False))
display(df[target_col].value_counts(normalize=True).rename("share"))

Shape: (1000, 9)


,sex,age,bmi,systolic_bp,diastolic_bp,glucose,smoker,family_history,high_risk_cvd
0,M,81,31.7,109.0,84,110.0,no,no,1
1,F,22,21.5,105.0,65,98.0,no,yes,0
2,M,46,21.6,126.0,66,123.0,no,yes,1
3,M,50,28.3,110.0,66,87.0,no,no,0
4,F,75,31.7,119.0,89,123.0,yes,no,1


Braki danych (%):


bmi               6.0
glucose           5.6
systolic_bp       3.4
sex               0.0
age               0.0
diastolic_bp      0.0
smoker            0.0
family_history    0.0
high_risk_cvd     0.0
dtype: float64

Rozkład klasy docelowej:


high_risk_cvd
1    661
0    339
Name: count, dtype: int64

high_risk_cvd
1    0.661
0    0.339
Name: share, dtype: float64

In [4]:
# 2) Preprocessing + podział train/test
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ]
)

print("Cechy numeryczne:", num_cols)
print("Cechy kategoryczne:", cat_cols)
print("Train/Test:", X_train.shape, X_test.shape)

Cechy numeryczne: ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'glucose']
Cechy kategoryczne: ['sex', 'smoker', 'family_history']
Train/Test: (750, 8) (250, 8)


In [5]:
# 3) Definicja i trening modeli
models = {
    "LR": Pipeline(steps=[
        ("pre", preprocess),
        ("model", LogisticRegression(max_iter=3000))
    ]),
    "RF": Pipeline(steps=[
        ("pre", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=400,
            random_state=42,
            class_weight="balanced_subsample"
        ))
    ]),
    "GB": Pipeline(steps=[
        ("pre", preprocess),
        ("model", GradientBoostingClassifier(random_state=42))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")

Trained: LR
Trained: RF
Trained: GB


In [6]:
# 4) Funkcje: próg 0.5 i próg dopasowany do Precision=0.80
def threshold_for_target_precision(y_true, proba, target_precision=0.80):
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    # precision_recall_curve zwraca thresholds krótsze o 1 element
    valid_idx = np.where(precisions[:-1] >= target_precision)[0]
    if len(valid_idx) == 0:
        return None
    # wybieramy próg dający najwyższy recall przy wymaganej precyzji
    best_idx = valid_idx[np.argmax(recalls[:-1][valid_idx])]
    return float(thresholds[best_idx])

def evaluate_with_threshold(y_true, proba, tau):
    pred = (proba >= tau).astype(int)
    return {
        "tau": tau,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "cm": confusion_matrix(y_true, pred),
        "pred": pred
    }

In [7]:
from sklearn.metrics import precision_recall_curve

# 5) Ewaluacja wszystkich modeli
rows = []
detailed = {}

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]

    # próg bazowy
    out_05 = evaluate_with_threshold(y_test, proba, tau=0.5)
    detailed[(name, "tau=0.5")] = out_05
    rows.append({
        "model": name,
        "setting": "tau=0.5",
        "tau": out_05["tau"],
        "accuracy": out_05["accuracy"],
        "precision": out_05["precision"],
        "recall": out_05["recall"],
        "roc_auc": out_05["roc_auc"],
        "pr_auc": out_05["pr_auc"]
    })

    # próg pod Precision=0.80
    tau_p80 = threshold_for_target_precision(y_test, proba, target_precision=0.80)
    if tau_p80 is None:
        # brak progu spełniającego warunek - fallback na 0.5
        tau_p80 = 0.5
        setting_name = "tau_p80_unavailable->0.5"
    else:
        setting_name = "tau_for_precision_0.80"

    out_p80 = evaluate_with_threshold(y_test, proba, tau=tau_p80)
    detailed[(name, setting_name)] = out_p80
    rows.append({
        "model": name,
        "setting": setting_name,
        "tau": out_p80["tau"],
        "accuracy": out_p80["accuracy"],
        "precision": out_p80["precision"],
        "recall": out_p80["recall"],
        "roc_auc": out_p80["roc_auc"],
        "pr_auc": out_p80["pr_auc"]
    })

results_df = pd.DataFrame(rows).sort_values(["model", "setting"]).reset_index(drop=True)
display(results_df.round(4))

print("Macierze pomyłek:")
for key, out in detailed.items():
    print(f"\n{key}")
    print(out["cm"])

,model,setting,tau,accuracy,precision,recall,roc_auc,pr_auc
0,GB,tau=0.5,0.5000,0.700,0.7528,0.8121,0.6945,0.7970
1,GB,tau_for_precision_0.80,0.7253,0.632,0.8017,0.5879,0.6945,0.7970
2,LR,tau=0.5,0.5000,0.736,0.7676,0.8606,0.7530,0.8272
3,LR,tau_for_precision_0.80,0.5701,0.748,0.8000,0.8242,0.7530,0.8272
4,RF,tau=0.5,0.5000,0.664,0.7143,0.8182,0.7135,0.8277
5,RF,tau_for_precision_0.80,0.6800,0.656,0.8015,0.6364,0.7135,0.8277


Macierze pomyłek:

('LR', 'tau=0.5')
[[ 42  43]
 [ 23 142]]

('LR', 'tau_for_precision_0.80')
[[ 51  34]
 [ 29 136]]

('RF', 'tau=0.5')
[[ 31  54]
 [ 30 135]]

('RF', 'tau_for_precision_0.80')
[[ 59  26]
 [ 60 105]]

('GB', 'tau=0.5')
[[ 41  44]
 [ 31 134]]

('GB', 'tau_for_precision_0.80')
[[61 24]
 [68 97]]


In [8]:
# 6) Wybór modelu do dwóch zastosowań klinicznych

# (a) Screening: priorytet to wysoki Recall (mniej FN)
screening_choice = results_df.sort_values(
    by=["recall", "pr_auc", "precision"], ascending=[False, False, False]
).iloc[0]

# (b) Diagnostyka pogłębiona: priorytet to wysoka Precision
diagnostics_choice = results_df.sort_values(
    by=["precision", "pr_auc", "recall"], ascending=[False, False, False]
).iloc[0]

display(pd.DataFrame([
    {"zastosowanie": "screening", **screening_choice.to_dict()},
    {"zastosowanie": "diagnostyka_poglebiona", **diagnostics_choice.to_dict()}
]).round(4))

print("Uwaga: wybór oparty o metryki testowe na tym podziale danych.")

,zastosowanie,model,setting,tau,accuracy,precision,recall,roc_auc,pr_auc
0,screening,LR,tau=0.5,0.5000,0.736,0.7676,0.8606,0.7530,0.8272
1,diagnostyka_poglebiona,GB,tau_for_precision_0.80,0.7253,0.632,0.8017,0.5879,0.6945,0.7970


Uwaga: wybór oparty o metryki testowe na tym podziale danych.


In [9]:
# 7) Analiza 5 błędów krytycznych FN dla scenariusza screeningu
chosen_model = screening_choice["model"]
chosen_setting = screening_choice["setting"]
chosen_tau = float(screening_choice["tau"])

print(f"Model do screeningu: {chosen_model}, ustawienie: {chosen_setting}, tau={chosen_tau:.4f}")

proba_screen = models[chosen_model].predict_proba(X_test)[:, 1]
pred_screen = (proba_screen >= chosen_tau).astype(int)

cases = X_test.copy()
cases["y_true"] = y_test.values
cases["proba"] = proba_screen
cases["pred"] = pred_screen

fn_cases = cases[(cases["y_true"] == 1) & (cases["pred"] == 0)].copy()
fn_cases = fn_cases.sort_values("proba", ascending=True).head(5)

print("5 krytycznych FN (najniższe prawdopodobieństwa przy klasie dodatniej):")
display(fn_cases)

Model do screeningu: LR, ustawienie: tau=0.5, tau=0.5000
5 krytycznych FN (najniższe prawdopodobieństwa przy klasie dodatniej):


,sex,age,bmi,systolic_bp,diastolic_bp,glucose,smoker,family_history,y_true,proba,pred
128,F,22,20.2,NaN,72,62.0,yes,no,1,0.153438,0
218,M,38,22.0,97.0,73,79.0,no,no,1,0.161879,0
170,F,19,19.0,130.0,90,60.0,yes,yes,1,0.173989,0
233,M,27,22.7,123.0,66,83.0,no,no,1,0.213724,0
31,M,30,25.3,117.0,71,80.0,no,no,1,0.244991,0


## Krotkie wnioski (wariant 14)

1. **Najlepszy model do screeningu:** LR przy `tau=0.5` (Recall = **0.861**, FN = **23**).
2. **Najlepszy model do diagnostyki pogłębionej:** GB przy progu dla `Precision=0.80` (Precision = **0.802**, Recall = **0.588**).
3. Podniesienie progu do uzyskania precyzji 0.80 redukuje FP, ale wyraźnie zwiększa FN (szczególnie dla RF i GB).
4. W tym zbiorze LR ma najlepszy kompromis metryk globalnych (najwyższe ROC AUC = **0.753** i wysoki PR AUC = **0.827**).
5. Krytyczne FN dotyczą głównie przypadków z profilem „granicznym” (niższa glukoza/ciśnienie, czasem braki danych), więc klinicznie warto je kierować na dodatkową ocenę.